**Data Preprocessing:**

In [104]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [105]:
df=pd.read_csv('anime.csv')

In [106]:
df.shape

(12294, 7)

In [107]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [108]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [109]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [110]:
#checking missing values
df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [111]:
#handling missing values
df["genre"] = df["genre"].fillna(df["genre"].mode()[0])
df["type"]=df["type"].fillna(df["type"].mode()[0])
df['rating'] = df['rating'].fillna(df['rating'].median())

In [112]:
#checking missing values
df.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,0
episodes,0
rating,0
members,0


In [113]:
# checking unique values
df['type'].unique()

array(['Movie', 'TV', 'OVA', 'Special', 'Music', 'ONA'], dtype=object)

In [114]:
df['genre'].nunique()

3264

In [115]:
# checking duplicates
df.duplicated().sum()

np.int64(0)

In [116]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


**Feature Extraction:**

In [117]:
# selected features for recommendation
features = ['genre' , 'type' , 'rating' , 'members']

In [118]:
from sklearn.preprocessing import OneHotEncoder , MinMaxScaler

In [119]:
# Convert Categorical Features into Numerical Representation
encoder = OneHotEncoder()
categorical_data = encoder.fit_transform(df[['genre', 'type']]).toarray()

In [120]:
# Convert encoded data to dataframe
encoded_df = pd.DataFrame(categorical_data,columns=encoder.get_feature_names_out(['genre', 'type']))

In [121]:
# # Normalize Numerical Features
scaler = MinMaxScaler()
numerical_data = scaler.fit_transform(df[['rating', 'members']])

In [122]:
# Convert scaled data to dataframe
scaled_df = pd.DataFrame(numerical_data,columns=['rating', 'members'])

In [123]:
# Combine Features
final_data = np.hstack((categorical_data, numerical_data))
print("Final Feature Shape:")
print(final_data.shape)

Final Feature Shape:
(12294, 3272)


In [124]:
# Pivot Table
pivot_table = pd.pivot_table(data=df,index='anime_id',values='rating',aggfunc='mean')
print(pivot_table.head())

          rating
anime_id        
1           8.82
5           8.40
6           8.32
7           7.36
8           7.06


In [125]:
# Combine All Features
final_data = pd.concat([pivot_table.reset_index(drop=True),encoded_df,scaled_df],axis=1)
final_data.head()


,rating,genre_Action,"genre_Action, Adventure","genre_Action, Adventure, Cars, Comedy, Sci-Fi, Shounen","genre_Action, Adventure, Cars, Mecha, Sci-Fi, Shounen, Sports","genre_Action, Adventure, Cars, Sci-Fi","genre_Action, Adventure, Comedy","genre_Action, Adventure, Comedy, Demons, Drama, Ecchi, Horror, Mystery, Romance, Sci-Fi","genre_Action, Adventure, Comedy, Demons, Fantasy, Magic","genre_Action, Adventure, Comedy, Demons, Fantasy, Magic, Romance, Shounen, Supernatural",...,genre_Vampire,genre_Yaoi,type_Movie,type_Music,type_ONA,type_OVA,type_Special,type_TV,rating,members
0,8.82,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.924370,0.197872
1,8.40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.911164,0.782770
2,8.32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.909964,0.112689
3,7.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.900360,0.664325
4,7.06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.899160,0.149186


**Recommendation System:**

In [126]:
# Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(final_data)
similarity_matrix.shape

(12294, 12294)

In [127]:
# Convert Similarity Matrix to DataFrame
similarity_df = pd.DataFrame(similarity_matrix,index=df['anime_id'],columns=df['anime_id'])

In [128]:
# Remove Self Similarity
np.fill_diagonal(similarity_df.values,0)

In [129]:
# Recommendation Function
def recommend_anime(anime_name,top_n=10,threshold=0.5):
    # Get Anime ID
    anime_id = df[df['name'] == anime_name]['anime_id'].values
    # Anime Not Found
    if len(anime_id) == 0:
        return "Anime not found"
    anime_id = anime_id[0]
    # Similarity Scores
    sim_scores = similarity_df[anime_id]
    # Apply Threshold
    sim_scores = sim_scores[sim_scores >= threshold]
    # Sort Similarity Scores
    sim_scores = sim_scores.sort_values(ascending=False)
    # Top Similar Anime IDs
    top_anime_ids = sim_scores.head(top_n).index
    # Recommended Anime
    recommendations = df[df['anime_id'].isin(top_anime_ids)][['anime_id','name','genre','type','rating','members']]
    return recommendations

In [130]:
# Example Recommendation
print(recommend_anime(anime_name='Naruto',top_n=10,threshold=0.5))

     anime_id                              name  \
1        5114  Fullmetal Alchemist: Brotherhood   
22          1                      Cowboy Bebop   
23      30276                     One Punch Man   
118     19815                   No Game No Life   
131      4224                         Toradora!   
159      6547                      Angel Beats!   
200       121               Fullmetal Alchemist   
445     10620                  Mirai Nikki (TV)   
615      1735                Naruto: Shippuuden   
804     11757                  Sword Art Online   

                                                 genre type  rating  members  
1    Action, Adventure, Drama, Fantasy, Magic, Mili...   TV    9.26   793665  
22     Action, Adventure, Comedy, Drama, Sci-Fi, Space   TV    8.82   486824  
23   Action, Comedy, Parody, Sci-Fi, Seinen, Super ...   TV    8.82   552458  
118  Adventure, Comedy, Ecchi, Fantasy, Game, Super...   TV    8.47   602291  
131             Comedy, Romance, School, Sl

In [131]:
# Experiment with Different Threshold Values
print("Threshold = 0.3")
print(recommend_anime(anime_name='Naruto',threshold=0.3))
print("Threshold = 0.7")
print(recommend_anime(anime_name='Naruto',threshold=0.7))

Threshold = 0.3
     anime_id                              name  \
1        5114  Fullmetal Alchemist: Brotherhood   
22          1                      Cowboy Bebop   
23      30276                     One Punch Man   
118     19815                   No Game No Life   
131      4224                         Toradora!   
159      6547                      Angel Beats!   
200       121               Fullmetal Alchemist   
445     10620                  Mirai Nikki (TV)   
615      1735                Naruto: Shippuuden   
804     11757                  Sword Art Online   

                                                 genre type  rating  members  
1    Action, Adventure, Drama, Fantasy, Magic, Mili...   TV    9.26   793665  
22     Action, Adventure, Comedy, Drama, Sci-Fi, Space   TV    8.82   486824  
23   Action, Comedy, Parody, Sci-Fi, Seinen, Super ...   TV    8.82   552458  
118  Adventure, Comedy, Ecchi, Fantasy, Game, Super...   TV    8.47   602291  
131             Comedy, Rom

In [132]:
# Most Similar Anime IDs
print("Most Similar Anime IDs")
print(similarity_df.idxmax()[0:10])

Most Similar Anime IDs
anime_id
32281     7311
5114     19815
28977    15417
9253     30484
9969     28977
32935    28891
11061      136
820       3665
15335    21899
15417    28977
dtype: int64


In [133]:
# Fetch Anime Details
print("Anime Details")
print(df[(df.anime_id == 20) | (df.anime_id == 1735)][['name','genre','rating']])

Anime Details
                   name                                              genre  \
615  Naruto: Shippuuden  Action, Comedy, Martial Arts, Shounen, Super P...   
841              Naruto  Action, Comedy, Martial Arts, Shounen, Super P...   

     rating  
615    7.94  
841    7.81  


**Analyzing the performance of the recommendation system and identify areas of improvement :**

**The recommendation system effectively suggests similar anime using cosine similarity based on genre, type, ratings, and members features.**

**Lower threshold values provide more recommendations, while higher thresholds improve recommendation relevance and accuracy.**

**The system can be improved by adding user watch history, collaborative filtering, and anime reviews for more personalized recommendations.**

**Interview Questions:**

**1. Can you explain the difference between user-based and item-based collaborative filtering?**

**User-based collaborative** filtering recommends items based on similar users’ preferences. If two users have similar ratings, items liked by one user are recommended to the other.

**Item-based collaborative** filtering recommends items similar to the ones a user already liked by comparing item similarities instead of user similarities.

**2. What is collaborative filtering, and how does it work?**

**Collaborative filtering** is a recommendation technique that suggests items based on user interactions, ratings, or preferences. It works by identifying similarities between users or items and recommending products, movies, or anime that similar users liked or similar items related to previous choices.